# Walmart Sales RAG with LangChain, FAISS, and GPT-4o-mini

A beginner-friendly Retrieval-Augmented Generation (RAG) pipeline over the Walmart sales CSV.

**Stack:** LangChain, FAISS, OpenAI embeddings, and `gpt-4o-mini`.


## 1. Install dependencies

In [ ]:
%pip install -q -U langchain langchain-openai langchain-community faiss-cpu pandas python-dotenv

In [ ]:
%pip install pandas

## 2. Configure the OpenAI API key

Set `OPENAI_API_KEY` as an environment variable or put it in a `.env` file:

```text
OPENAI_API_KEY=your_api_key_here
```


In [ ]:
OPENAI_API_KEY=''

## 3. Load the Walmart dataset

Download the CSV from Kaggle and place it next to this notebook.

Change `CSV_PATH` if your file has a different name.


In [ ]:
import pandas as pd

CSV_PATH = "Walmart.csv"

df = pd.read_csv(CSV_PATH)
df = df.dropna(how="all").copy()

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.head())
print(df.columns.tolist())

## 4. Convert structured rows into RAG documents

In [ ]:
from langchain_core.documents import Document

def row_to_text(row):
    date = row["Date"].strftime("%Y-%m-%d") if pd.notna(row["Date"]) else "Unknown"
    holiday = "holiday week" if row["Holiday_Flag"] == 1 else "non-holiday week"

    return (
        f"Store {int(row['Store'])} recorded weekly sales of "
        f"${row['Weekly_Sales']:,.2f} on {date}. "
        f"This was a {holiday}. "
        f"The temperature was {row['Temperature']:.2f} degrees Fahrenheit, "
        f"the fuel price was {row['Fuel_Price']:.2f}, "
        f"the CPI was {row['CPI']:.2f}, "
        f"and unemployment was {row['Unemployment']:.2f}%."
    )

documents = [
    Document(
        page_content=row_to_text(row),
        metadata={
            "store": int(row["Store"]),
            "date": row["Date"].strftime("%Y-%m-%d") if pd.notna(row["Date"]) else None,
            "holiday_flag": int(row["Holiday_Flag"]),
            "weekly_sales": float(row["Weekly_Sales"]),
        },
    )
    for _, row in df.iterrows()
]

print(f"Created {len(documents):,} documents.")
print(documents[0].page_content)
print(documents[0].metadata)

## 5. Create OpenAI embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("Embedding model initialized.")

## 6. Build the FAISS vector store

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)

print("FAISS vector store created.")

## 7. Create the retriever

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

print("Retriever created.")

## 8. Test retrieval before involving the LLM

In [ ]:
query = "Which Walmart stores had high sales during holiday weeks?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Retrieved document {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)
    print()

## 9. Initialize GPT-4o-mini

This uses the requested configuration:

```python
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
```


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
)

print("GPT-4o-mini initialized.")

## 10. Create the RAG prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Walmart sales data analyst.

Answer the user's question using ONLY the Walmart records provided
in the retrieved context.

Rules:
1. Do not invent facts or values.
2. If the context is insufficient, say so.
3. When useful, mention the relevant store and date.
4. For numerical questions, show calculations when possible.
5. Keep the answer concise and clear.

Retrieved context:
{context}
""",
    ),
    ("human", "{question}"),
])

## 11. Build the RAG chain

Pipeline:

`Question → FAISS Retriever → Retrieved Context → Prompt → GPT-4o-mini → Answer`


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[Document {i}] {doc.page_content}"
        for i, doc in enumerate(docs, 1)
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created.")

## 12. Ask your first RAG question

In [ ]:
question = "Which store had the highest sales among the retrieved records?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

## 13. Inspect retrieved documents and the final answer

In [ ]:
def ask_rag(question, k=5):
    docs = vector_store.similarity_search(question, k=k)
    context = format_docs(docs)

    response = llm.invoke(
        prompt.invoke({
            "context": context,
            "question": question,
        })
    )

    return {
        "question": question,
        "answer": response.content,
        "documents": docs,
    }

result = ask_rag(
    "Which Walmart stores had high weekly sales during holiday weeks?",
    k=5,
)

print("ANSWER")
print("=" * 80)
print(result["answer"])

print("\nRETRIEVED DOCUMENTS")
print("=" * 80)

for i, doc in enumerate(result["documents"], 1):
    print(f"\n[{i}] {doc.page_content}")
    print("Metadata:", doc.metadata)

## 14. Try multiple questions

In [ ]:
questions = [
    "Which stores had high weekly sales during holiday weeks?",
    "What happened to sales when unemployment was relatively high?",
    "Which retrieved record has the highest weekly sales?",
    "Compare the sales of the retrieved holiday-week records.",
]

for q in questions:
    result = ask_rag(q, k=5)
    print("=" * 100)
    print("QUESTION:", q)
    print("ANSWER:", result["answer"])
    print()

## 15. Save the FAISS index

This lets you reuse the vector store without recomputing embeddings.

Only load FAISS indexes that you trust because loading uses pickle-based deserialization.


In [ ]:
FAISS_PATH = "walmart_faiss"

vector_store.save_local(FAISS_PATH)
print(f"FAISS index saved to: {FAISS_PATH}")

## 16. Load the FAISS index later

In [ ]:
loaded_vector_store = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True,
)

loaded_retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

print("FAISS index loaded successfully.")

## 17. Reusable RAG function

In [ ]:
def ask(question, k=5):
    docs = loaded_vector_store.similarity_search(question, k=k)
    context = format_docs(docs)

    response = llm.invoke(
        prompt.invoke({
            "context": context,
            "question": question,
        })
    )

    return response.content

print(ask("Which retrieved record has the highest weekly sales?"))

# Suggested next experiments

1. Change `k` from 1 → 3 → 5 → 10 and inspect retrieval.
2. Try FAISS MMR retrieval with `search_type="mmr"`.
3. Add metadata filtering for a specific store.
4. Retrieve 20 documents and rerank them before sending 5 to GPT-4o-mini.
5. Add a pandas/SQL tool for exact numerical questions.
6. Build a hybrid RAG system combining semantic retrieval with structured analysis.
